# B 담당 Colab 실행 노트북 (딥 특징 추출 + 평균점 섞기)

실행 전용 노트북이다. 실험 로직은 `src/`의 `.py`에만 있고, 여기서는 환경 준비 · 스크립트 실행 · 결과 훑어보기만 한다.

- 런타임 유형을 **T4 GPU**로 먼저 바꾼 뒤 시작한다(중간에 바꾸면 `/content`가 비워져 데이터를 다시 복사해야 한다).
- 세션을 다시 열면 **셀 1~6을 다시 실행**하고 셀 7(진행 상태)부터 본다. 끝나지 않은 셀은 그대로 다시 실행하면 이어서 진행한다.
- test 분할은 D8 전까지 쓰지 않는다. 평균점 섞기는 `--split dev`만 실행한다.
- 평가는 `EVALUATION_POLICY.md`를 따른다. 믹스견은 Top-1 · Coverage, 순종은 Top-1 · Top-5를 따로 보고, 반복 r1~r3의 확률 평균을 대표값으로 쓴다.
- UF `hold`는 예측도 채점도 하지 않는다.
- DNA 표(`dna_UF`)는 `/content`에만 둔다. 세션을 다시 열면 채점 전에 셀 10-1의 변환 셀과 확인 셀을 다시 실행한다.
- 커밋 전에 출력을 지운다.

**셀 1. 설정 (채울 것)**: 저장소 주소, Drive 바로가기 경로 두 개를 채운다.

In [ ]:
REPO_URL = "https://github.com/kimminkyung05/puppy.."
DRIVE_DIR = "/content/drive/MyDrive/puppy..."
DATA_DIR  = "/content/drive/MyDrive/Tsinghua Dogs Dataset"
DATA_ZIPS = []   # 비워 두면 셀 3이 저해상도 이미지·주석 zip을 자동으로 찾음
LOCAL_DATA = "/content/data"
REPO = "/content/dog-mix-cv"
CODE = f"{REPO}/B"   # B 코드는 저장소의 B/ 폴더에 있다

**셀 2. Drive 연결, 폴더 확인**: 두 폴더가 보이는지 확인하고 데이터 폴더의 파일 목록을 출력한다.

In [ ]:
import os, glob
from google.colab import drive

# force_remount=True가 기존 마운트를 알아서 정리한다.
# /content/drive를 직접 지우지 않는다: 마운트가 살아 있으면 Drive의 실제 파일이 지워진다.
drive.mount('/content/drive', force_remount=True)

for d in [DRIVE_DIR, DATA_DIR]:
    assert os.path.isdir(d), (f"폴더가 없음: {d}\n"
        "→ 링크를 열고 '바로가기 추가 → 내 드라이브'를 했는지, 아래 목록에서 이름을 확인")
print("내 드라이브:", os.listdir("/content/drive/MyDrive"))
print("\n데이터 폴더 내용:")
for p in sorted(glob.glob(f"{DATA_DIR}/**/*", recursive=True))[:100]:
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:10.1f} MB  {os.path.relpath(p, DATA_DIR)}")

In [ ]:
import os

print(f"검증할 DRIVE_DIR: {DRIVE_DIR}")
if os.path.exists(DRIVE_DIR):
    print("[V] DRIVE_DIR 폴더가 정상적으로 존재합니다.")
    print("\n--- 하위 폴더 및 파일 목록 ---")
    try:
        sub_items = os.listdir(DRIVE_DIR)
        for item in sub_items:
            item_path = os.path.join(DRIVE_DIR, item)
            is_dir = "[폴더]" if os.path.isdir(item_path) else "[파일]"
            print(f"{is_dir} {item}")
            # evalset 폴더인 경우 내부 파일도 추가 출력
            if item == "evalset" and os.path.isdir(item_path):
                print("   └─ 하위 파일:", os.listdir(item_path))
    except Exception as e:
         print(f"[X] 하위 목록을 읽는 중 오류 발생: {e}")
else:
    print("[X] DRIVE_DIR 폴더가 존재하지 않습니다! 경로를 다시 확인해 주세요.")

**셀 3. 받을 zip 고르기**: 저해상도 이미지 · 주석 · 분할 zip만 고른다(고해상도 제외).

In [ ]:
if not DATA_ZIPS:
    zips = glob.glob(f"{DATA_DIR}/**/*.zip", recursive=True)
    DATA_ZIPS = sorted(
        z for z in zips
        if "high" not in os.path.basename(z).lower()
        and any(k in os.path.basename(z).lower() for k in ["low", "annotation", "split", "trainval"])
    )
print("복사할 zip:", *DATA_ZIPS, sep="\n  ")
assert DATA_ZIPS, "자동으로 못 찾음 → 셀 2 출력을 보고 셀 1의 DATA_ZIPS에 경로를 직접 적기"

**셀 4. `/content`로 복사 · 압축 해제**: 이번 세션에 이미 푼 zip은 건너뛴다. 마지막 줄에 풀린 폴더 이름이 나온다.

In [ ]:
import shutil, subprocess
os.makedirs(LOCAL_DATA, exist_ok=True)
for z in DATA_ZIPS:
    name = os.path.basename(z)
    done = f"{LOCAL_DATA}/.done_{name}"
    if os.path.exists(done):
        print("이미 풀림:", name); continue
    try:
        print("복사:", name); shutil.copy(z, f"/content/{name}")
    except OSError as e:
        raise RuntimeError(f"{name} 복사 실패: {e}\n"
            "→ 공개 폴더 다운로드 한도일 수 있음. 그 zip을 '사본 만들기'로 내 드라이브에 복사한 뒤 "
            "셀 1의 DATA_ZIPS에 그 경로를 적기") from e
    print("압축 해제:", name)
    subprocess.run(["unzip", "-q", "-o", f"/content/{name}", "-d", LOCAL_DATA], check=True)
    os.remove(f"/content/{name}"); open(done, "w").close()
print(os.listdir(LOCAL_DATA))

**셀 5. 코드 받기 + 경로 자동 설정**: 저장소를 clone/pull하고 `B/` 폴더로 이동한 뒤, 풀린 폴더를 찾아 `/content/paths_colab.yaml`을 만들어 `DOGMIX_PATHS`로 지정한다(`configs/paths.yaml`은 고치지 않는다).

저장소가 공개면 그대로 돈다. 비공개로 바꾸면 clone에 토큰이 필요하다: Colab 왼쪽 **열쇠 아이콘(보안 비밀)** → 이름 `GITHUB_TOKEN`, 값은 GitHub 읽기 전용 토큰(Fine-grained, 이 저장소 Contents: Read-only) → **노트북 액세스** 켜기. 토큰을 셀에 직접 적지 않는다.

In [ ]:
import re, yaml
CLASS_RE = re.compile(r"^\d+-n\d+-")
def find_class_root(base, want_xml):
    for dirpath, dirnames, _ in os.walk(base):
        cls = [d for d in dirnames if CLASS_RE.match(d)]
        if len(cls) >= 100:
            sample = os.listdir(os.path.join(dirpath, cls[0]))[:50]
            if any(f.endswith(".xml") for f in sample) == want_xml:
                return dirpath
    return None
def find_file(base, name):
    for dirpath, _, files in os.walk(base):
        if name in files:
            return os.path.join(dirpath, name)
    return None
img_root = find_class_root(LOCAL_DATA, want_xml=False)
ann_root = find_class_root(LOCAL_DATA, want_xml=True)
train_lst, val_lst = find_file(LOCAL_DATA, "train.lst"), find_file(LOCAL_DATA, "validation.lst")
if not (train_lst and val_lst):
    !git clone -q https://github.com/dejungle/Tsinghua-Dogs /content/tsinghua-split
    train_lst = "/content/tsinghua-split/split/train.lst"
    val_lst   = "/content/tsinghua-split/split/validation.lst"
# 비공개 저장소: Colab 왼쪽 열쇠 아이콘(보안 비밀)에 GITHUB_TOKEN 을 넣고 "노트북 액세스"를 켠다 (읽기 전용 토큰)
# 토큰은 clone/pull 할 때만 쓰고 .git/config 에 남기지 않는다. 출력에 나오면 *** 로 가린다
from google.colab import userdata
try:
    GH_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GH_TOKEN = None
AUTH_URL = REPO_URL.replace("https://", f"https://x-access-token:{GH_TOKEN}@") if GH_TOKEN else REPO_URL
HIDE = 'sed "s/x-access-token:[^@]*@/x-access-token:***@/g"'
if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q {AUTH_URL} {REPO} 2>&1 | {HIDE}
    if os.path.isdir(f"{REPO}/.git"):
        !git -C {REPO} remote set-url origin {REPO_URL}
else:
    !git -C {REPO} pull -q {AUTH_URL} main 2>&1 | {HIDE}
assert os.path.isdir(CODE), (f"코드를 못 받음: {CODE} 없음 → 저장소가 비공개면 보안 비밀 GITHUB_TOKEN 을 넣었는지, "
                             "노트북 액세스를 켰는지, 토큰에 이 저장소 Contents 읽기 권한이 있는지 확인")
%cd {CODE}
cfg = yaml.safe_load(open("configs/paths.yaml", encoding="utf-8"))
cfg["tsinghua"].update(image_root=img_root, annotation_root=ann_root, train_list=train_lst, val_list=val_lst)
cfg["evalset"]["index_csv"] = f"{DRIVE_DIR}/evalset/eval_index.csv"
cfg["outputs"].update(features_root=f"{DRIVE_DIR}/features", preds_root=f"{DRIVE_DIR}/preds", logs_root=f"{DRIVE_DIR}/logs")
with open("/content/paths_colab.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)
os.environ["DOGMIX_PATHS"] = "/content/paths_colab.yaml"
print(open("/content/paths_colab.yaml", encoding="utf-8").read())
missing = [k for k, v in {"이미지": img_root, "주석": ann_root, "train.lst": train_lst, "validation.lst": val_lst}.items()
           if not v or not os.path.exists(v)]
assert not missing, f"못 찾음: {missing} → 셀 4의 폴더 목록을 보고 paths_colab.yaml을 직접 고치기"

**셀 6. GPU 확인, 패키지 설치, Hugging Face 로그인**: DINOv3 접근 승인을 받은 계정의 토큰으로 로그인한다.

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt
from huggingface_hub import login; login()

**셀 6-1. 데이터 · 모델 점검**: 경로, `bodybndbox` 태그, 몸통 박스를 못 찾은 비율, 클래스 수(127), 모델 ID, 출력 차원, DINOv3 `num_register_tokens`를 출력한다. '확인 필요'가 있으면 추출 전에 해결한다.

In [ ]:
!python -m src.common.check_data
!python -m src.features.check_models

**셀 7. 진행 상태 확인**: 세션을 다시 열면 셀 1~6 실행 후 여기부터 보고, '완료'가 아닌 것을 이어서 실행한다.

In [ ]:
!python -m src.common.status --logs 5

**셀 8. 첫 실행 확인 (1,000장 시간 재기)**: 샤드가 '완료'이고 `tmp_files`가 0인지(Drive에서 임시 파일 이름 바꾸기가 되는지) 보고, 장/초로 전체 시간을 추정한다.

In [ ]:
!python -m src.features.extract_deep --backbone dinov3 --subset train --limit 1000 --tag timing
!python -m src.common.status --logs 1 --eta

**셀 9. 특징 추출**: dinov3 → dinov2 → clip 순서로 train · val · eval. 끊기면 같은 셀을 다시 실행한다(끝난 샤드는 건너뜀).

In [ ]:
# 정식 평가셋(A의 eval_index.csv)이 아직 없으면 eval 은 건너뛴다. UF 평가셋은 셀 10-2 에서 따로 뽑는다
HAS_EVAL = os.path.exists(f"{DRIVE_DIR}/evalset/eval_index.csv")
print("정식 평가셋:", "있음 → eval 도 뽑는다" if HAS_EVAL else "없음 → train · val 만 (eval 은 건너뜀)")
for b in ["dinov3", "dinov2", "clip"]:
    for s in ["train", "val"] + (["eval"] if HAS_EVAL else []):
        !python -m src.features.extract_deep --backbone {b} --subset {s}

**셀 10. 평균점 섞기 (dev)**: 추출기 4종 × L2 없음/있음. 이미 있는 예측은 건너뛴다. `sift`는 C의 특징이 올라온 뒤에 돌아간다.

In [ ]:
# 정식 평가셋 특징(features/{b}/eval)이 있는 추출기만 돌린다. 없으면 건너뛰어 실패 로그를 남기지 않는다
for b in ["dinov3", "dinov2", "clip", "sift"]:
    if not os.path.exists(f"{DRIVE_DIR}/features/{b}/eval/index.csv"):
        print(f"[건너뜀] {b}: 정식 평가셋 특징 없음 (features/{b}/eval)")
        continue
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --skip-existing --l2

**셀 10-1. UF 초안 평가셋 확인**: Top-1 정답용 `dna_UF`는 변환 셀이 `/content/uf_private`에만 저장한다. UF 공개 자료(믹스견 97 + 순종 3)의 칸 사진 200장(개마다 얼굴 · 전신)과 인덱스가 Drive에 있는지 본다. 정식 평가셋(A의 `eval_index.csv`, D의 SAM 3 크롭)이 아닌 임시 초안이라 특징 이름은 `eval_uf_panels`다. 사진은 화면에 띄우지 않는다(UF 이용 조건: 교육용 저장만 허용).

In [ ]:
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

try:
    # 구글 드라이브 시트에 접근하기 위한 사용자 인증
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # 'eval_index_UF_panels_colab' 스프레드시트 열기
    sheet_name = 'eval_index_UF_panels_colab'
    print(f"구글 스프레드시트 '{sheet_name}'에서 데이터를 가져오는 중입니다...")
    sh = gc.open(sheet_name)
    worksheet = sh.get_worksheet(0)

    # 데이터를 판다스 데이터프레임으로 변환
    data = worksheet.get_all_values()
    df_sheet = pd.DataFrame(data[1:], columns=data[0])

    # 타겟 CSV 파일 경로에 저장
    target_csv_path = f"{DRIVE_DIR}/evalset/eval_index_UF_panels.csv"
    df_sheet.to_csv(target_csv_path, index=False, encoding='utf-8')
    print(f"[V] 성공적으로 스프레드시트를 다운로드하여 {target_csv_path} 경로에 CSV로 변환 저장했습니다!")
    print("이제 위에 있는 '셀 10-1 (uf_code_check)'을 다시 실행해 보세요!")
except Exception as e:
    print(f"[X] 변환 중 오류가 발생했습니다: {e}")
    print("만약 권한 에러나 시트를 찾을 수 없다는 에러가 발생하면, 구글 드라이브에서 'eval_index_UF_panels_colab' 파일 이름을 찾아 이 Colab에 로그인한 계정에 공유되어 있는지, 파일명에 오타가 없는지 확인해 주세요. (평가셋은 링크 공개로 바꾸지 않습니다)")

In [ ]:
import os, glob
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default

UF_DRAFT = f"{DRIVE_DIR}/evalset/uf_draft"
UF_PRIVATE = "/content/uf_private"                   # 개별 DNA 표는 Drive에 복사하지 않고 세션 안에만 둔다
os.makedirs(UF_PRIVATE, exist_ok=True)
WANTED = ["truth_UF", "breed_map_UF_draft", "dna_UF"]   # 필요한 CSV 이름(확장자 제외)

# 폴더가 없으면 자동으로 생성해 줍니다.
if not os.path.exists(UF_DRAFT):
    os.makedirs(UF_DRAFT, exist_ok=True)
    print(f"[안내] {UF_DRAFT} 폴더가 없어서 새로 생성했습니다.")
else:
    print("uf_draft 폴더 목록:", os.listdir(UF_DRAFT))

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

for name in WANTED:
    target = f"{UF_PRIVATE if name == 'dna_UF' else UF_DRAFT}/{name}.csv"
    if os.path.exists(target):
        print(f"[건너뜀] 이미 있음 → {target}")
        continue
    # 폴더 안의 스프레드시트 이름을 찾는다 (예: truth_UF.gsheet, truth_UF.csv.gsheet)
    found = [os.path.basename(p)[:-len(".gsheet")] for p in glob.glob(f"{UF_DRAFT}/*{name}*.gsheet") or glob.glob(f"{DRIVE_DIR}/evalset/*{name}*.gsheet")]
    if not found:
        # 만약 gsheet 파일이 로컬 캐싱에 안보인다면, gspread로 직접 열기를 바로 시도합니다.
        found = [name]
    try:
        print(f"스프레드시트 '{found[0]}'에서 데이터를 가져오는 중...")
        ws = gc.open(found[0]).get_worksheet(0)
        # 화면에 보이는 값이 아니라 원래 값을 가져온다 (소수 자릿수가 잘리지 않게)
        data = ws.get_all_values(value_render_option="UNFORMATTED_VALUE")
        df = pd.DataFrame(data[1:], columns=data[0])
        df.to_csv(target, index=False, encoding="utf-8")
        print(f"[V] {found[0]} → {target} | {len(df)}행 × {len(df.columns)}열")
    except Exception as e:
        print(f"[X] {name} 변환 실패: {e}")

print("\n변환 후 폴더 내용:", os.listdir(UF_DRAFT))

In [ ]:
import pandas as pd
UF_DIR = f"{DRIVE_DIR}/evalset"
UF_INDEX = f"{UF_DIR}/eval_index_UF_panels.csv"
UF_DNA = "/content/uf_private/dna_UF.csv"
UF_BACKBONES = ["dinov3", "dinov2", "clip"]   # 셀 7에서 train 특징이 '완료'인 추출기만 남긴다
# sift는 여기서 뽑지 않는다. C의 특징(train · eval_uf_panels)이 올라와 있으면 평균점 섞기에만 넣는다
UF_PROTO = UF_BACKBONES + (["sift"] if all(os.path.exists(f"{DRIVE_DIR}/features/sift/{s}/index.csv")
                                           for s in ["train", "eval_uf_panels"]) else [])
print("평균점 섞기 대상:", UF_PROTO)
idx = pd.read_csv(UF_INDEX)
print(f"칸 사진 {len(glob.glob(f'{UF_DIR}/raw/UF/panels/*.jpg'))}장 (기대 200) | 인덱스 {len(idx)}행, "
      f"개 {idx['dog_id'].nunique()}마리 | split {idx['split'].value_counts().to_dict()}")
for p in [idx["path"].iloc[0], idx["path"].iloc[-1]]:
    print("[있음]" if os.path.exists(p) else "[없음]", p)
missing = [p for p in idx["path"] if not os.path.exists(p)]
assert not missing, f"사진 {len(missing)}장을 못 찾음 → 인덱스의 path 앞부분이 DRIVE_DIR과 같은지 확인: {missing[:3]}"
for f in ["truth_UF.csv", "breed_map_UF_draft.csv"]:
    print("[있음]" if os.path.exists(f"{UF_DIR}/uf_draft/{f}") else "[없음]", f"{UF_DIR}/uf_draft/{f}")
assert os.path.exists(UF_DNA), "dna_UF.csv 가 없음 → 바로 위 변환 셀을 다시 실행"
n_dna = len(pd.read_csv(UF_DNA))
assert n_dna == 370, f"dna_UF 가 {n_dna}줄(기대 370) → 시트가 바뀌었는지 확인"
print("[있음]", UF_DNA, f"{n_dna}줄")

**셀 10-2. UF 특징 추출**: 칸 사진 200장을 `eval_uf_panels`로 저장한다. 끊기면 같은 셀을 다시 실행한다.

In [ ]:
for b in UF_BACKBONES:
    !python -m src.features.extract_deep --backbone {b} --subset eval --eval-index "{UF_INDEX}" --save-as eval_uf_panels

**셀 10-3. UF 평균점 섞기 (dev)**: 임시 dev 20마리만 돌린다(hold는 예측하지 않는다). L2 없음/있음 × r1~r3, 이미 있는 예측은 건너뛴다. 얼굴 · 전신 두 장의 확률은 개 단위로 평균해 저장한다(평가 기준 6절). `sift`는 C의 특징이 있을 때만 들어간다. 실행 이름은 `RQ1-eval_uf_panels-{추출기}-proto-r{반복}`.

In [ ]:
for b in UF_PROTO:
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --eval-subset eval_uf_panels --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --eval-subset eval_uf_panels --skip-existing --l2

**셀 10-4. UF 임시 채점 + 상태 확인 (dev)**: 지표 정의는 `docs/METRICS_FINAL.md`(A 최종 정의 복사본)를 따른다. 매핑은 정확만 쓴다.

- 첫 표 Top-1(분류 보조 지표): r1~r3 확률 평균 뒤 채점. 공동 1위는 그중 하나를 맞히면 정답
- 반복별 최소~최대, Coverage, 정답/평가 가능/전체
- 그 아래: 주 지표 원래 비율 포착@3, 보조 지표 재정규화 포착@3 · 재정규화 분포 겹침, 참고값 원래 비율 분포 겹침. 정확 + 유사 매핑 Top-1 등 참고값은 `--reference`를 붙이면 나온다.

매핑 초안 기준의 임시 채점이다. 정식 채점이 아니며 A의 `eval.py`가 나오면 그것으로 바꾼다.

In [ ]:
!python scripts/provisional_score_uf.py --uf-dir "{UF_DIR}/uf_draft" --eval-subset eval_uf_panels --dna "{UF_DNA}"
!python -m src.common.status --logs 5

**셀 10-5. 순종 보조 평가 (Tsinghua val, dev 절반)**: 순종 사진을 순종으로 답하는지 본다. 믹스견 평가와 따로 보고한다. 먼저 val을 클래스별로 반씩 나눈 임시 분할(`evalset/val_split_seed1.csv`, 이미 있으면 건너뜀)을 만들고, 그 dev 쪽 2,540장만 평가한다. 평균점은 train에서만 만든다. `{추출기}/train`과 `{추출기}/val` 특징이 '완료'인 추출기만 돌아간다. `sift`는 두 특징이 모두 있을 때만 들어간다. test 2,540장은 D8 전까지 쓰지 않는다.

In [ ]:
!python scripts/make_val_split.py
PB_PROTO = ["dinov3", "dinov2", "clip"] + (["sift"] if all(os.path.exists(f"{DRIVE_DIR}/features/sift/{s}/index.csv")
                                                             for s in ["train", "val"]) else [])
for b in PB_PROTO:
    !python -m src.rq1.prototype_fcls --backbone {b} --eval-kind purebred --repeats 1 2 3 --split dev --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --eval-kind purebred --repeats 1 2 3 --split dev --skip-existing --l2

**셀 10-6. 순종 임시 채점 (dev 절반)**: 첫 표가 `EVALUATION_POLICY.md`의 순종 지표(Top-1 · Top-5)이고, r1~r3 확률 평균이 대표값이다. 믹스견 표와 합치지 않는다. 정답은 사진에 붙은 품종 라벨이며 DNA가 아니다. 정식 채점이 아니다.

In [ ]:
!python scripts/provisional_score_purebred.py --uf-dir "{DRIVE_DIR}/evalset/uf_draft"

**셀 10-7. 두 부모 평균점 그림 (dev)**: DNA 상위 두 품종이 모두 정확 매핑되고 합이 75% 이상인 dev 개(4마리)가 두 부모 품종 평균점을 잇는 선분 어디에 놓이는지 그린다. 평균점은 학습셋 전체 특징(`train`)으로 만들고, B 예측(r1~r3 평균) 위치도 함께 표시한다.

- 특징과 예측을 읽기만 한다. `extract`는 실행하지 않는다(로컬 CPU용).
- 개별 개의 DNA 비율이 그림에 들어간다. 기본은 `/content/uf_private/figs`에만 둔다. `SAVE_TO_DRIVE = True`로 바꾸면 Drive `figures_private/`에도 복사한다. Drive 폴더 공유가 "제한됨"(팀원만)인지 먼저 확인한다. 저장소에는 넣지 않는다. 내려받으려면 마지막 줄 주석을 푼다.
- hold는 쓰지 않는다.

In [ ]:
SAVE_TO_DRIVE = False   # Drive 폴더 공유를 "제한됨"(팀원만)으로 바꾼 뒤 True. 개별 DNA가 든 그림이다
!apt-get -qq install -y fonts-nanum > /dev/null
FIG_DIR = "/content/uf_private/figs"
PP_ARGS = (f'--backbones dinov3 dinov2 clip --train-subset train --eval-subset eval_uf_panels '
           f'--dna "{UF_DNA}" --breed-map "{UF_DIR}/uf_draft/breed_map_UF_draft.csv" --eval-index "{UF_INDEX}" '
           f'--preds-dir "{DRIVE_DIR}/preds/dev" --out-dir "{FIG_DIR}"')
!python scripts/parent_projection.py plot {PP_ARGS}
import shutil
from IPython.display import Image, display
figs = sorted(glob.glob(f"{FIG_DIR}/dev_parent_projection_*"))
for f in figs:
    if f.endswith(".png"):
        print(f); display(Image(f, width=900))
if SAVE_TO_DRIVE:
    SAVE_DIR = f"{DRIVE_DIR}/figures_private"   # 저장소(GitHub)에는 넣지 않는다
    os.makedirs(SAVE_DIR, exist_ok=True)
    for f in figs:
        shutil.copy2(f, SAVE_DIR)              # 같은 이름이 있으면 새 결과로 바꾼다
        print("[Drive 저장]", f"{SAVE_DIR}/{os.path.basename(f)}")
else:
    print("Drive에 저장하지 않음 (SAVE_TO_DRIVE = False)")
# from google.colab import files; [files.download(f) for f in figs]

**셀 11. 마무리**: 최종 상태를 보고 Drive에 쓰기를 끝낸 뒤 연결을 끊는다.

In [ ]:
import os
from google.colab import drive

# Drive가 이미 끊겨 있으면 status가 "폴더 없음"으로 잘못 보이므로 먼저 확인한다.
if not os.path.ismount("/content/drive"):
    print("Drive가 연결되어 있지 않음 (이미 마무리했거나 셀 2를 안 돌림).\n"
          "→ 결과를 보려면 셀 1, 2를 다시 실행한 뒤 이 셀을 실행한다. 결과 파일은 Drive에 그대로 있다.")
else:
    !python -m src.common.status --logs 10
    # 이 줄 뒤로는 Drive가 끊긴다. 이후 셀을 더 돌리려면 셀 1, 2부터 다시 실행한다.
    drive.flush_and_unmount()